# AI Agente Analyze Fraud Rules 
Mich y cami
Ts
ops_fraud.falcon_declined_transactions fdt
Chargebacks.
Analytics_bi.transactions

User feedback: tabla, declined transactions.

Archivos PLT (v 2.0)

Diario -> dependemos de mich. 
Moverlo a transactions_ytd. 
Agregar 3ds a la tabla


transactions_


Order of Agents to implement
1. Detector de patrón: diario/semanal
- Suggestion: 
- Revise CB 

2. Agente de validación de reglas 
- Sheets - fraud rules


3. Revisión de CB, suggestion de nuevas falcon 
- reglas/limitantes de falcon
- Muy definido el output.
- Use this information de user feedback and suggest a change. 


In [1]:
cd ..

/Users/camila.cusicanqui/Documents/GitHub/frod-agentic-ai


In [13]:
from utils.data_ingest import get_db_conn
import pandas as pd
import numpy as np
import gspread
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, timedelta, datetime
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format
import gspread
from dateutil.utils import today



In [3]:
import os
import yaml

with open(".config/credentials-mage.yaml", "r") as f:
    creds = yaml.safe_load(f)

for item in creds:
    os.environ[item["name"]] = str(item["value"])

In [4]:
import sys
sys.path.append('/Users/camila.cusicanqui/Documents/GitHub/analytics-mage-infra/mage-fraud-space')

In [5]:
import logging

# Set up a basic logger
logger = logging.getLogger("fraud_context")
logger.setLevel(logging.INFO)  

In [6]:
fraud_rules_query = f"""
SELECT user_id, klrid, transaction_id, amount, timestamp_mx_created_at, prosa_timestamp, merchant, mcc_code, regla, pos_entry_mode, pin_capabilities, card_type, product_type
FROM ops_fraud.falcon_declined_transactions;
"""

In [7]:
fraud_rules_df = pd.read_sql_query(fraud_rules_query, get_db_conn())

/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_86335/1460050075.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fraud_rules_df = pd.read_sql_query(fraud_rules_query, get_db_conn())


In [8]:
fraud_rules_df.head()

,user_id,klrid,transaction_id,amount,timestamp_mx_created_at,prosa_timestamp,merchant,mcc_code,regla,pos_entry_mode,pin_capabilities,card_type,product_type
0,431c8363-067d-4786-9fa1-e0e7614a5f4f,97ace885-4cd1-4952-938f-49b520a3a17d,PARABILIUM:124767681,-288.99,2025-08-05 00:00:42.912,2025-08-05 00:00:43.144,EBANX 1TIKTOK SH ARTSCIUDAD DE MEXDF MX,7311,VE___Decline_Excessive_Velocity,CNP Manual,unknown,PHYSICAL,CREDIT_5456_BIN
1,52507396-3c60-4c60-987c-2547c1f06885,b5847975-7c66-4864-993e-8315acc9e40c,PARABILIUM:124768652,-300.00,2025-08-05 00:07:08.305,2025-08-05 00:07:08.607,NUVEIMXCALIENTE TIJUANA BCNMX,7941,FV___High_Fraud_Score_Online,CNP Card On File,unknown,VIRTUAL,CREDIT_5401_BIN
2,52507396-3c60-4c60-987c-2547c1f06885,b5847975-7c66-4864-993e-8315acc9e40c,PARABILIUM:124768690,-300.00,2025-08-05 00:07:24.646,2025-08-05 00:07:24.905,NUVEIMXCALIENTE TIJUANA BCNMX,7941,FV___High_Fraud_Score_Online,CNP Card On File,unknown,VIRTUAL,CREDIT_5401_BIN
3,52507396-3c60-4c60-987c-2547c1f06885,b5847975-7c66-4864-993e-8315acc9e40c,PARABILIUM:124768837,-200.00,2025-08-05 00:08:19.412,2025-08-05 00:08:19.657,NUVEIMXCALIENTE TIJUANA BCNMX,7941,FV___High_Fraud_Score_Online,CNP Card On File,unknown,VIRTUAL,CREDIT_5401_BIN
4,0da0f6a0-af55-4fdc-8e12-8a54da213262,73810457-6f0d-463d-abae-2261d89cc2b9,PARABILIUM:124768884,-227.32,2025-08-05 00:08:37.951,2025-08-05 00:08:38.305,D LOCALTDA TEMU MEXICO DF MX,5969,FV___High_Fraud_Score_Online,CNP Manual,unknown,VIRTUAL,CREDIT_5456_BIN


In [9]:
SERVICE_ACCOUNT_FILE = r'.config/klar-cami-cusi.json'
gc = gspread.service_account(filename=SERVICE_ACCOUNT_FILE)
sheet_id = "1cQ8QzQqml1oBdYliYpW1V2ZquHJrH5iBZzwlGnYFV5I"
spreadsheet = gc.open_by_key(sheet_id)
# connect to ghseets
worksheet = spreadsheet.worksheet('All Falcon Rules')
falcon_rules_df = pd.DataFrame(worksheet.get_all_records())

# PROSA Rules prompt
TODO: 
- explain in prompt what each rule is and what it does,
- explain PROSA dynamics and how it works with the rules, and how the rules are applied in the transactions
- Each rule in the 'All Falcon Rules' worksheet represents a specific fraud prevention rule implement into Falcon, PROSA's fraud detection system. 
- These rules are designed to prevent fraudulent transactions by utilizing various factors and utilizing counter numeric variables.

Each rule has the following characteristics:
- Rule ID: an identifier to group similar rules.
- Rule name: identifier + general description of the rule.
- Logic: detailed rule definition.
- Rule type: the rules usually repeat certain general patterns. 
- Amount limit: if applies for rule, the limit of the amount implemented. 
- Score limit: each transaction in PROSA has a score. If this rule includes a threshold for the score, the limit of the amount implemented.
- Transaction limit: if applies to rule, the limit of how many transactions a user, or list can have. 
- Timeframe: timeeframe for which the rule applies. For example, if a merchant has a card validation merchant of GOOGLE and 24 hours later a TELCEL intent then the transaction does not pass. 
- Lists: There are lists that the rule can use. For example, in the case of MOContador it checks whether a card is inside a archivo de embozo.
- UDV: Each rule has a numeric rule, the numeric rules contain user defined variables that can store numbers (counter), dates, true or False. 
- Segment: The card portfolio group it belongs to.
- Subsegment: subsegment of the segment. 
- Motive: rule was created because of a recent fraud incident, or are general preventative rules.
- Last updated date: date of last update. 
- Creation date: date of rule creation.
- Estado: state of rule. 
- Comments: additional rule comments.

# Tools for pattern detector agent. 

We're going to prioritize the pattern detector agent. We run a weekly analysis of rules and it's motives for declining so, we need to check the following:
- Week by week what are the increases in declinations per rule
- Who are the merchants that are causing these casualites? 
- What's the behavior of the transaction like? Does the user have previous purchaes with this merchant? In velocity cases are they 
    - Is it multiples tries per user and then the trx passes?
    -
- Are there simlar behaviors between the declined transactions? 
- What are pos entry modes of the declined transactions? 
- What segment does the client belong to ?
- What are the stats of the merchant in the last year? last month ? last 7days? Do they significantly differ from the transactions that we declined?
- Does the merchant usually have 3ds, cvv present, not present? 
- Does the merchant currently have CB or active CB? 


# TODO:
- analyze mage analytics infra and see what class we can create for the tools agent manager 

In [10]:
from fraud_utils.ai_agents.fraud_alerts import MerchantFraudContextBuilder

In [11]:
fraud_rules_df["amount_abs"] = fraud_rules_df["amount"].abs()

In [33]:

# Get today's date
today = datetime.today()

# Calculate the start of the current week (Monday as the start of the week)
current_week_start = today - timedelta(days=today.weekday())

# Generate a list of week start dates for the last 7 days and earlier weeks
fraud_rules_df["trx_start_week"] = fraud_rules_df["timestamp_mx_created_at"].apply(
    lambda x: (current_week_start - timedelta(weeks=(current_week_start - x).days // 7)).date()
)
fraud_rules_df["trx_start_week"] = pd.to_datetime(fraud_rules_df["trx_start_week"])

In [26]:
fraud_rules_df[fraud_rules_df["timestamp_mx_created_at"] >= '2026-04-01'].trx_start_week.value_counts()

trx_start_week
2026-04-06    6685
2026-04-13    5456
2026-04-20    4444
2026-05-11    3673
2026-05-04    3059
2026-04-27    2689
2026-05-18    1485
Name: count, dtype: int64

In [55]:
week_analysis = fraud_rules_df.groupby(["trx_start_week", "regla"]).agg(
    transaction_count=("transaction_id", "nunique"),
    total_amount=("amount_abs", "sum")
).reset_index()
# Build a complete week x rule grid so missing weeks are explicit zeros
week_analysis["trx_start_week"] = pd.to_datetime(week_analysis["trx_start_week"]).dt.normalize()

all_weeks = pd.date_range(
    start=week_analysis["trx_start_week"].min(),
    end=week_analysis["trx_start_week"].max(),
    freq="7D"
)
all_rules = week_analysis["regla"].dropna().unique()

full_index = pd.MultiIndex.from_product(
    [all_rules, all_weeks],
    names=["regla", "trx_start_week"]
)

week_analysis = (
    week_analysis
    .set_index(["regla", "trx_start_week"])
    .reindex(full_index, fill_value=0)
    .reset_index()
    .sort_values(["regla", "trx_start_week"])
)

# Week-over-week pct change by rule
pct_cols = week_analysis.groupby("regla")[["transaction_count", "total_amount"]].pct_change().mul(100)
week_analysis["transaction_count_pct_diff"] = pct_cols["transaction_count"]
week_analysis["total_amount_pct_diff"] = pct_cols["total_amount"]

# Clean infinite values from division by zero and round for readability
week_analysis[["transaction_count_pct_diff", "total_amount_pct_diff"]] = (
    week_analysis[["transaction_count_pct_diff", "total_amount_pct_diff"]]
    .round(2)
)


In [68]:
week_observation =  str((current_week_start - timedelta(weeks=1)).date())
week_observation

'2026-05-11'

In [50]:
week_analysis[(week_analysis["trx_start_week"] >= '2026-05-01')
            #    & (week_analysis["transaction_count_pct_diff"] < 0)
               & (week_analysis["regla"] == 'MC_Decline_MCC_Limit_HR')]

,regla,trx_start_week,transaction_count,total_amount,transaction_count_pct_diff,total_amount_pct_diff
2498,MC_Decline_MCC_Limit_HR,2026-05-04,17,"48,738.87",-61.36,-24.81
2499,MC_Decline_MCC_Limit_HR,2026-05-11,28,"45,004.16",64.71,-7.66
2500,MC_Decline_MCC_Limit_HR,2026-05-18,21,"61,674.00",-25.00,37.04


In [58]:
week_analysis = week_analysis[
    (week_analysis.transaction_count_pct_diff.notna()) | (week_analysis.total_amount_pct_diff.notna())
].copy()

In [71]:
week_analysis[
    ((week_analysis["transaction_count_pct_diff"] > 0)
    | (week_analysis["total_amount_pct_diff"] > 0))
    & (week_analysis["trx_start_week"] == week_observation)
]

,regla,trx_start_week,transaction_count,total_amount,transaction_count_pct_diff,total_amount_pct_diff
2827,AV_Decline_Acquirer_CVV_B2C,2026-05-11,9,"9,318.90",inf,inf
2335,FV_Decline_High_Fraud_Score_Online_ALL,2026-05-11,2159,"4,233,311.56",27.53,34.44
367,HF_High_Fraud_Score_General_ALL,2026-05-11,244,"492,090.05",10.41,5.84
3032,KQ_Decline_LumepicKrispy_ALL,2026-05-11,3,568.00,200.00,468.00
2376,LC_Decline_LowScore_CardVerifications_HR,2026-05-11,96,96.00,24.68,24.68
2499,MC_Decline_MCC_Limit_HR,2026-05-11,28,"45,004.16",64.71,-7.66
2868,SN_Decline_Cashout_Validation_B2C,2026-05-11,1,747.09,-50.00,3.54
2950,SQ_Decline_CashOut_Telcel_ALL,2026-05-11,617,"247,010.67",44.16,131.16
1638,VE_Decline_Excessive_Velocity_B2C,2026-05-11,333,"64,318.01",12.50,-12.71
3073,WQ_Decline_WalTaquilla_ALL,2026-05-11,2,"9,113.60",inf,inf


In [73]:
week_analysis[
    ((week_analysis["transaction_count_pct_diff"] > 0)
    | (week_analysis["total_amount_pct_diff"] > 0))
    & (week_analysis["trx_start_week"] == week_observation)
]

,regla,trx_start_week,transaction_count,total_amount,transaction_count_pct_diff,total_amount_pct_diff
2827,AV_Decline_Acquirer_CVV_B2C,2026-05-11,9,"9,318.90",inf,inf
2335,FV_Decline_High_Fraud_Score_Online_ALL,2026-05-11,2159,"4,233,311.56",27.53,34.44
367,HF_High_Fraud_Score_General_ALL,2026-05-11,244,"492,090.05",10.41,5.84
3032,KQ_Decline_LumepicKrispy_ALL,2026-05-11,3,568.00,200.00,468.00
2376,LC_Decline_LowScore_CardVerifications_HR,2026-05-11,96,96.00,24.68,24.68
2499,MC_Decline_MCC_Limit_HR,2026-05-11,28,"45,004.16",64.71,-7.66
2868,SN_Decline_Cashout_Validation_B2C,2026-05-11,1,747.09,-50.00,3.54
2950,SQ_Decline_CashOut_Telcel_ALL,2026-05-11,617,"247,010.67",44.16,131.16
1638,VE_Decline_Excessive_Velocity_B2C,2026-05-11,333,"64,318.01",12.50,-12.71
3073,WQ_Decline_WalTaquilla_ALL,2026-05-11,2,"9,113.60",inf,inf
